# Tenacious-Bench — SimPO Judge Training (Colab T4)

**Week 11 · Day 5 · Path B (preference-tuned judge)**

Trains a Qwen 3.5 4B LoRA judge on the 965 preference pairs in [training_data/](training_data/) using SimPO (Meng et al., NeurIPS 2024).

**Pre-requisites (must be done before running this notebook):**

- Day 0 smoke test ([test.ipynb](test.ipynb)) passed end to end.
- HF write token in Colab Secrets as `HF_TOKEN`.
- Repo cloned with `training_data/train.jsonl` and `training_data/dev.jsonl` present.

**Expected wall time:** 30–90 min on T4. If not converging by 30 min, **kill it and check the data** — do not throw more compute (per challenge brief §Act IV).

**Cost:** $0 on Colab T4. RunPod 4090 ≈ $0.34/hr (cap $5).

**Expected folder layout in `/content/tenacious-bench/`:**

```
training/
  train_judge.py          ← training entrypoint
  colab_smoke_test.py     ← Day 0 smoke test
  requirements.txt        ← pinned deps (trl>=0.12.0)
  checkpoints/            ← created by this run (best by dev pairwise acc)
  training_run.log        ← loss + dev-acc every 100 steps
  training_summary.json   ← final hyperparams, wall time, cost
training_data/
  train.jsonl             ← 618 preference pairs
  dev.jsonl               ← 347 preference pairs
  build_cost_log.json     ← cost ledger (data-prep + training)
tenacious_bench_v0.1/
  train/  dev/  held_out/ ← source partitions (held_out is sealed)
```


## Step 1 — Check GPU


In [ ]:
import subprocess, torch

print(
    subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
    or "NO GPU — change runtime to T4"
)
print(f"torch={torch.__version__}  cuda={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(
        f"GPU: {torch.cuda.get_device_name(0)}  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB"
    )
    print(f"bf16 supported: {torch.cuda.is_bf16_supported()}")

## Step 2 — Clone repo (skip if already cloned)


In [ ]:
import os
REPO_URL = "https://github.com/<your-fork>/tenacious-bench.git"  # ← edit me
if not os.path.isdir("/content/tenacious-bench"):
    !git clone $REPO_URL /content/tenacious-bench
%cd /content/tenacious-bench
!wc -l training_data/train.jsonl training_data/dev.jsonl

## Step 3 — Install dependencies


In [ ]:
!pip install -q unsloth
!pip install -q "trl>=0.12.0" "peft>=0.12.0" "datasets>=2.20.0" "accelerate>=0.33.0" \
               sentencepiece protobuf "huggingface_hub>=0.24.0"
import trl, peft, transformers
print(f"trl={trl.__version__}  peft={peft.__version__}  transformers={transformers.__version__}")

## Step 4 — HuggingFace authentication

Add a write-scope token at Colab sidebar → 🔑 → `HF_TOKEN`.


In [ ]:
from huggingface_hub import login

login()

## Step 5 — Core training run (γ=1.0, γ/β=0.5)

This is the **headline run**. SimPO with γ=1.0, β=2.0 → γ/β=0.5, the best-region from the SimPO paper's Table 3 (Mistral-Base / Llama-3-8B-Instruct). Runs `training/train_judge.py` which:

- Loads `unsloth/Qwen3.5-4B` in 16-bit + LoRA (rank=16, α=32).
- Trains on `training_data/train.jsonl` (618 pairs).
- Evaluates dev pairwise accuracy every 100 steps; keeps the best checkpoint.
- Logs to `training/training_run.log`.
- Pushes the LoRA adapter to HuggingFace (private).
- Appends a cost record to `training_data/build_cost_log.json`.

**Replace `<your-hf-user>`.** Drop `--hf-repo` for a local-only run.


In [ ]:
HF_USER = "<your-hf-user>"  # ← edit me
!python training/train_judge.py \
    --gamma 1.0 \
    --platform colab \
    --hf-repo $HF_USER/tenacious-judge-qwen35-4b

## Step 6 — Ablation run (γ=1.5, γ/β=0.75)

Per `methodology_rationale.md`: γ=1.5 (γ/β=0.75) is the upper-edge SimPO ablation variant. Both γ values are reported in `ablations/ablation_results.json` on Day 6.

Skip if Step 5 ate your Colab session quota — re-attach a fresh runtime and run this alone.


In [ ]:
!python training/train_judge.py \
    --gamma 1.5 \
    --platform colab \
    --hf-repo $HF_USER/tenacious-judge-qwen35-4b-gamma15

## Step 7 — Inspect loss curve & training summary


In [ ]:
import json, pathlib

summary = json.loads(pathlib.Path("training/training_summary.json").read_text())
print(json.dumps(summary, indent=2))

In [ ]:
# Plot loss curve from training_run.log
import re, matplotlib.pyplot as plt

log = pathlib.Path("training/training_run.log").read_text()
steps, losses = [], []
for m in re.finditer(r"step=(\d+).*?loss=([0-9.]+)", log):
    steps.append(int(m.group(1)))
    losses.append(float(m.group(2)))
if steps:
    plt.figure(figsize=(8, 4))
    plt.plot(steps, losses)
    plt.xlabel("step")
    plt.ylabel("SimPO loss")
    plt.title("Training loss")
    plt.grid(alpha=0.3)
    plt.show()
else:
    print("No step= lines parsed — check training/training_run.log manually")

## Step 8 — Cost summary

Total spend across the week (data prep + training). Must stay under **$10** per the challenge brief's cost-discipline observable.


In [ ]:
import json, pathlib
from collections import defaultdict

log = json.loads(pathlib.Path("training_data/build_cost_log.json").read_text())
totals = defaultdict(float)
for r in log:
    totals[r.get("bucket", "other")] += float(r.get("cost_usd", 0.0))
print(f"{'Bucket':<22} {'USD':>8}")
print("-" * 32)
for b, c in sorted(totals.items()):
    print(f"{b:<22} {c:>8.4f}")
print("-" * 32)
print(f"{'TOTAL':<22} {sum(totals.values()):>8.4f}  /  $10.00 budget")

## Step 9 — Commit artifacts back to the repo

Save the run log + summary + checkpoints metadata so the repo's `training/` folder reflects this run. The LoRA adapter weights stay on HF Hub (not committed to git).

Run locally after downloading from Colab, or use git-from-Colab if you have your push token wired up.


In [ ]:
!ls -lah training/training_run.log training/training_summary.json training_data/build_cost_log.json

**Next:** Day 6 → ablations on the sealed held-out (`ablations/ablation_results.json`, Delta A/B/cost-Pareto) and the HF dataset + model-card publication.
